# 10 — Noise Burst Robustness

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

Notebook 10 stress-tests the control stack under measurement bursts, drift jumps, and CGCS boundary pressure.

## Purpose

Previous notebooks showed:

- Kalman dominates smooth drift regimes.
- MPC helps only when bounded and constrained.
- Joint estimation helps when Ω/B coupling matters.

This notebook asks:

> What happens when calibration measurements contain bursts and outliers?

## Policies compared

- none
- moving average
- scalar Kalman
- joint Kalman
- coupled MPC
- robust gated Kalman
- oracle

## Main idea

Ordinary Kalman assumes measurement noise is well behaved.  
Burst noise violates that assumption.

A robust Kalman gate downweights suspicious measurements:

```text
if |measurement − prediction| > gate_threshold:
    downweight measurement
```

## Principle

Robust control requires detecting when calibration evidence is unreliable.


## Robust gating model

For a standard Kalman update:

```text
innovation = measurement − prediction
```

If the innovation is too large, the measurement may be an outlier.

Robust update:

```text
if |innovation| > gate_threshold:
    R_effective = R × inflation
else:
    R_effective = R
```

This makes the filter trust the prediction more during measurement bursts.

CGCS stability is tracked with:

```text
cos(θ) ≥ 1 / √(1² + 1²)
```


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures/noise_burst_robustness"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_fig(name):
    """Save figure as PNG and PDF using the Notebook 10 naming convention."""
    plt.savefig(f"{FIG_DIR}/10_noise_burst_robustness_{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{FIG_DIR}/10_noise_burst_robustness_{name}.pdf", bbox_inches="tight")


In [ ]:
def rabi_model(t, A, Omega, B):
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def phase_lock_metric(signal, target):
    dot = np.dot(signal, target)
    norm = np.linalg.norm(signal) * np.linalg.norm(target)
    if norm == 0:
        return np.nan
    return dot / norm


def moving_average(x, window=4):
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


def kalman_filter_1d(z, q, r, x0=None, p0=1.0):
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        x_pred = x
        p_pred = p + q

        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * (measurement - x_pred)
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, p_hist, k_hist


def robust_kalman_filter_1d(z, q, r, gate_threshold, inflation=80.0, x0=None, p0=1.0):
    """1D Kalman filter with outlier-gated measurement covariance inflation."""
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)
    gated = np.zeros_like(z, dtype=bool)
    innovation_hist = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        x_pred = x
        p_pred = p + q

        innovation = measurement - x_pred
        innovation_hist[i] = innovation

        r_eff = r
        if abs(innovation) > gate_threshold:
            r_eff = r * inflation
            gated[i] = True

        k_gain = p_pred / (p_pred + r_eff)
        x = x_pred + k_gain * innovation
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, p_hist, k_hist, gated, innovation_hist


def kalman_filter_joint(z, F, H, Q, R, x0=None, P0=None):
    z = np.asarray(z, dtype=float)
    n_steps = z.shape[0]
    n_state = F.shape[0]

    if x0 is None:
        x = z[0].reshape(n_state, 1)
    else:
        x = np.asarray(x0, dtype=float).reshape(n_state, 1)

    if P0 is None:
        P = np.eye(n_state)
    else:
        P = np.asarray(P0, dtype=float)

    x_filt = np.zeros((n_steps, n_state))
    K_trace = np.zeros((n_steps, n_state, H.shape[0]))

    I = np.eye(n_state)

    for i in range(n_steps):
        x_prior = F @ x
        P_prior = F @ P @ F.T + Q

        y = z[i].reshape(-1, 1) - H @ x_prior
        S = H @ P_prior @ H.T + R
        K = P_prior @ H.T @ np.linalg.inv(S)

        x = x_prior + K @ y
        P = (I - K @ H) @ P_prior

        x_filt[i] = x.ravel()
        K_trace[i] = K

    return x_filt, K_trace


def coupled_mpc_smoother(delta_omega_est, delta_b_est, alpha=0.82, omega_bound=0.03, b_bound=0.012):
    """Bounded smoother used as MPC-like controller baseline."""
    delta_omega_est = np.asarray(delta_omega_est)
    delta_b_est = np.asarray(delta_b_est)

    omega_cmd = np.zeros_like(delta_omega_est)
    b_cmd = np.zeros_like(delta_b_est)

    omega_cmd[0] = delta_omega_est[0]
    b_cmd[0] = delta_b_est[0]

    for k in range(1, len(delta_omega_est)):
        d_omega = np.clip(delta_omega_est[k] - omega_cmd[k-1], -omega_bound, omega_bound)
        d_b = np.clip(delta_b_est[k] - b_cmd[k-1], -b_bound, b_bound)

        omega_cmd[k] = omega_cmd[k-1] + alpha * d_omega
        b_cmd[k] = b_cmd[k-1] + alpha * d_b

    return omega_cmd, b_cmd


def evaluate_policy(name, delta_omega_hat, delta_b_hat, true_delta_omega, true_delta_b):
    omega_controlled = target_Omega + true_delta_omega - delta_omega_hat
    b_controlled = np.clip(target_B + true_delta_b - delta_b_hat, 0, 0.3)

    target_signal_local = rabi_model(pulse_t, target_A, target_Omega, target_B)

    response_rmse = []
    phase_lock = []
    controlled_signals = []

    for k in range(len(true_delta_omega)):
        y = rabi_model(pulse_t, target_A, omega_controlled[k], b_controlled[k])
        controlled_signals.append(y)
        response_rmse.append(rmse(y, target_signal_local))
        phase_lock.append(phase_lock_metric(y, target_signal_local))

    response_rmse = np.array(response_rmse)
    phase_lock = np.array(phase_lock)
    controlled_signals = np.array(controlled_signals)

    return {
        "name": name,
        "omega_command": np.asarray(delta_omega_hat),
        "b_command": np.asarray(delta_b_hat),
        "omega_controlled": omega_controlled,
        "b_controlled": b_controlled,
        "omega_error": omega_controlled - target_Omega,
        "b_error": b_controlled - target_B,
        "omega_rmse": rmse(omega_controlled, np.full(len(omega_controlled), target_Omega)),
        "b_rmse": rmse(b_controlled, np.full(len(b_controlled), target_B)),
        "response_rmse": response_rmse,
        "response_rmse_mean": float(np.mean(response_rmse)),
        "response_rmse_max": float(np.max(response_rmse)),
        "phase_lock": phase_lock,
        "min_phase_lock": float(np.min(phase_lock)),
        "mean_phase_lock": float(np.mean(phase_lock)),
        "max_angle_degrees": float(np.max(np.degrees(np.arccos(np.clip(phase_lock, -1, 1))))),
        "cgcs_margin_min": float(np.min(phase_lock - (1 / np.sqrt(2)))),
        "controlled_signals": controlled_signals,
    }


## Simulate burst-noise drift environment

The true drift is smooth/coupled, but measurements include:

- Gaussian measurement noise
- burst windows
- isolated outliers
- a small physical drift jump

This separates real drift from measurement corruption.


In [ ]:
n_blocks = 64
block_times = np.arange(n_blocks)

n_points_per_block = 140
pulse_t = np.linspace(0, 10, n_points_per_block)

target_A = 0.88
target_Omega = 2.45
target_B = 0.06

target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

# Smooth coupled true drift
shared = 0.016 * np.sin(2 * np.pi * block_times / 27 + 0.35)
omega_private = (
    0.045 * np.sin(2 * np.pi * block_times / 19 + 0.2)
    + 0.012 * np.sin(2 * np.pi * block_times / 7)
)
b_private = (
    0.0010 * block_times
    + 0.0045 * np.sin(2 * np.pi * block_times / 13 + 0.25)
)

# Small physical jump after block 42
jump = np.zeros(n_blocks)
jump[42:] += 0.018

true_delta_Omega = omega_private + shared + jump
true_delta_B = b_private + 0.22 * shared + 0.003 * (block_times >= 42)

Omega_uncomp = target_Omega + true_delta_Omega
B_uncomp = target_B + true_delta_B
A_uncomp = target_A + 0.012 * np.sin(2 * np.pi * block_times / 17)

# Measurement burst specification
burst_window = np.arange(24, 32)
outlier_blocks = np.array([11, 37, 51])

print(f"n_blocks = {n_blocks}")
print(f"true Ω/B drift correlation = {np.corrcoef(true_delta_Omega, true_delta_B)[0,1]:.3f}")
print(f"burst window = {burst_window[0]}..{burst_window[-1]}")
print(f"outlier blocks = {outlier_blocks.tolist()}")


## Generate corrupted calibration measurements


In [ ]:
Y_obs = []
Y_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, A_uncomp[k], Omega_uncomp[k], B_uncomp[k])

    sigma = 0.04
    if k in burst_window:
        sigma = 0.105

    y_obs = y_clean + sigma * np.random.randn(n_points_per_block)

    # Structured outlier: pulse-dependent bias, not just random noise.
    if k in outlier_blocks:
        y_obs += 0.09 * np.sin(1.7 * pulse_t + 0.5 * k)

    y_obs = np.clip(y_obs, 0, 1)

    Y_clean.append(y_clean)
    Y_obs.append(y_obs)

Y_clean = np.array(Y_clean)
Y_obs = np.array(Y_obs)

fit_params = []
fit_stds = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 5.0, 0.3])

for k in range(n_blocks):
    params, cov = fit_model(
        rabi_model,
        pulse_t,
        Y_obs[k],
        p0=initial_guess,
        bounds=bounds,
    )
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)

Omega_est = fit_params[:, 1]
B_est = fit_params[:, 2]

delta_Omega_est = Omega_est - target_Omega
delta_B_est = B_est - target_B

print("Corrupted calibration fits complete.")
print(f"measured Ω/B drift correlation = {np.corrcoef(delta_Omega_est, delta_B_est)[0,1]:.3f}")


## Define policies


In [ ]:
# Baselines
delta_Omega_none = np.zeros(n_blocks)
delta_B_none = np.zeros(n_blocks)

ma_window = 4
delta_Omega_ma = moving_average(delta_Omega_est, window=ma_window)
delta_B_ma = moving_average(delta_B_est, window=ma_window)

# Standard scalar Kalman
r_omega = float(np.median(fit_stds[:, 1] ** 2))
r_b = float(np.median(fit_stds[:, 2] ** 2))

q_omega = 8.0e-4
q_b = 1.0e-5

delta_Omega_scalar, _, _ = kalman_filter_1d(delta_Omega_est, q=q_omega, r=r_omega, p0=1.0)
delta_B_scalar, _, _ = kalman_filter_1d(delta_B_est, q=q_b, r=r_b, p0=1.0)

# Joint Kalman
z_joint = np.column_stack([delta_Omega_est, delta_B_est])

F = np.eye(2)
H = np.eye(2)

R = np.array([
    [r_omega, 0.0],
    [0.0, r_b],
])

q_cross = 0.35 * np.sqrt(q_omega * q_b)
Q_joint = np.array([
    [q_omega, q_cross],
    [q_cross, q_b],
])

joint_filt, K_joint = kalman_filter_joint(
    z_joint,
    F=F,
    H=H,
    Q=Q_joint,
    R=R,
    P0=np.eye(2),
)

delta_Omega_joint = joint_filt[:, 0]
delta_B_joint = joint_filt[:, 1]

# Robust gated Kalman
gate_omega = 0.028
gate_b = 0.016
inflation = 100.0

delta_Omega_robust, _, _, gated_omega, innovation_omega = robust_kalman_filter_1d(
    delta_Omega_est,
    q=q_omega,
    r=r_omega,
    gate_threshold=gate_omega,
    inflation=inflation,
    p0=1.0,
)

delta_B_robust, _, _, gated_b, innovation_b = robust_kalman_filter_1d(
    delta_B_est,
    q=q_b,
    r=r_b,
    gate_threshold=gate_b,
    inflation=inflation,
    p0=1.0,
)

# Coupled MPC smoother using robust estimate as base
delta_Omega_mpc, delta_B_mpc = coupled_mpc_smoother(
    delta_Omega_robust,
    delta_B_robust,
    alpha=0.82,
    omega_bound=0.025,
    b_bound=0.010,
)

# Oracle
delta_Omega_oracle = true_delta_Omega.copy()
delta_B_oracle = true_delta_B.copy()

print("Policies ready.")
print(f"Ω gated blocks: {np.where(gated_omega)[0].tolist()}")
print(f"B gated blocks: {np.where(gated_b)[0].tolist()}")


## Burst tracking: Ω


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true Ω drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, label="corrupted measured Ω")
plt.plot(block_times, delta_Omega_scalar, linewidth=2, label="scalar Kalman")
plt.plot(block_times, delta_Omega_joint, linewidth=2, label="joint Kalman")
plt.plot(block_times, delta_Omega_robust, linewidth=2, label="robust gated Kalman")
plt.axvspan(burst_window[0], burst_window[-1], alpha=0.15, label="burst window")
plt.scatter(outlier_blocks, delta_Omega_est[outlier_blocks], marker="x", s=90, label="outlier blocks")
plt.xlabel("calibration block")
plt.ylabel("Ω drift")
plt.title("Noise burst robustness: Ω burst tracking")
plt.legend()
plt.tight_layout()

save_fig("omega_burst_tracking")
plt.show()


## Burst tracking: B offset


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true B drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, label="corrupted measured B")
plt.plot(block_times, delta_B_scalar, linewidth=2, label="scalar Kalman")
plt.plot(block_times, delta_B_joint, linewidth=2, label="joint Kalman")
plt.plot(block_times, delta_B_robust, linewidth=2, label="robust gated Kalman")
plt.axvspan(burst_window[0], burst_window[-1], alpha=0.15, label="burst window")
plt.scatter(outlier_blocks, delta_B_est[outlier_blocks], marker="x", s=90, label="outlier blocks")
plt.xlabel("calibration block")
plt.ylabel("B drift")
plt.title("Noise burst robustness: B burst tracking")
plt.legend()
plt.tight_layout()

save_fig("offset_burst_tracking")
plt.show()


## Evaluate policies


In [ ]:
policies = {
    "none": (delta_Omega_none, delta_B_none),
    "moving_average": (delta_Omega_ma, delta_B_ma),
    "scalar_kalman": (delta_Omega_scalar, delta_B_scalar),
    "joint_kalman": (delta_Omega_joint, delta_B_joint),
    "robust_kalman": (delta_Omega_robust, delta_B_robust),
    "coupled_mpc": (delta_Omega_mpc, delta_B_mpc),
    "oracle": (delta_Omega_oracle, delta_B_oracle),
}

policy_results = {}

for name, (delta_omega_hat, delta_b_hat) in policies.items():
    policy_results[name] = evaluate_policy(
        name,
        delta_omega_hat,
        delta_b_hat,
        true_delta_Omega,
        true_delta_B,
    )

for name, result in policy_results.items():
    print(
        f"{name:18s} | "
        f"response RMSE mean = {result['response_rmse_mean']:.6f} | "
        f"max = {result['response_rmse_max']:.6f} | "
        f"min phase-lock = {result['min_phase_lock']:.6f}"
    )


## Response-level RMSE comparison


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    plt.plot(block_times, result["response_rmse"], marker="o", linewidth=1, label=name)

plt.axvspan(burst_window[0], burst_window[-1], alpha=0.15, label="burst window")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.title("Noise burst robustness: response RMSE comparison")
plt.legend()
plt.tight_layout()

save_fig("response_rmse_comparison")
plt.show()


## CGCS stability comparison


In [ ]:
threshold = 1 / np.sqrt(2)

plt.figure(figsize=(11, 5))
plt.axhline(threshold, linewidth=1, linestyle="--", label="45° threshold")

for name, result in policy_results.items():
    plt.plot(block_times, result["phase_lock"], marker="o", linewidth=1, label=name)

plt.axvspan(burst_window[0], burst_window[-1], alpha=0.15, label="burst window")
plt.ylim(0.68, 1.01)
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.title("Noise burst robustness: CGCS phase-lock stability")
plt.legend()
plt.tight_layout()

save_fig("cgcs_stability_comparison")
plt.show()


## Burst-window zoom


In [ ]:
zoom_lo = max(0, burst_window[0] - 3)
zoom_hi = min(n_blocks, burst_window[-1] + 4)
zoom = np.arange(zoom_lo, zoom_hi)

plt.figure(figsize=(11, 5))
for name in ["scalar_kalman", "joint_kalman", "robust_kalman", "coupled_mpc"]:
    plt.plot(zoom, policy_results[name]["response_rmse"][zoom], marker="o", linewidth=1, label=name)

plt.axvspan(burst_window[0], burst_window[-1], alpha=0.15, label="burst window")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target")
plt.title("Noise burst robustness: burst-window zoom")
plt.legend()
plt.tight_layout()

save_fig("burst_window_zoom")
plt.show()


## CGCS threshold margin


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    if name == "oracle":
        continue
    plt.plot(block_times, result["phase_lock"] - threshold, marker="o", linewidth=1, label=name)

plt.axhline(0, linewidth=1, linestyle="--", label="threshold margin = 0")
plt.axvspan(burst_window[0], burst_window[-1], alpha=0.15, label="burst window")
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.title("Noise burst robustness: threshold margin")
plt.legend()
plt.tight_layout()

save_fig("threshold_margin")
plt.show()


## Policy ranking


In [ ]:
ranking = sorted(policy_results.values(), key=lambda r: r["response_rmse_mean"])

policy_table = []
for rank, result in enumerate(ranking, start=1):
    policy_table.append({
        "rank": rank,
        "policy": result["name"],
        "omega_rmse": result["omega_rmse"],
        "b_rmse": result["b_rmse"],
        "response_rmse_mean": result["response_rmse_mean"],
        "response_rmse_max": result["response_rmse_max"],
        "min_phase_lock": result["min_phase_lock"],
        "mean_phase_lock": result["mean_phase_lock"],
        "max_angle_degrees": result["max_angle_degrees"],
        "cgcs_margin_min": result["cgcs_margin_min"],
        "all_blocks_phase_locked": bool(np.all(result["phase_lock"] >= threshold)),
    })

policy_table


In [ ]:
policy_names_ranked = [row["policy"] for row in policy_table]
response_means_ranked = [row["response_rmse_mean"] for row in policy_table]
response_max_ranked = [row["response_rmse_max"] for row in policy_table]

x = np.arange(len(policy_names_ranked))
width = 0.35

plt.figure(figsize=(11, 5))
bars_mean = plt.bar(x - width / 2, response_means_ranked, width, label="mean RMSE")
bars_max = plt.bar(x + width / 2, response_max_ranked, width, label="max RMSE")

plt.xticks(x, policy_names_ranked, rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Noise burst robustness: policy ranking")
plt.legend()

for bars in [bars_mean, bars_max]:
    for bar in bars:
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.002,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

plt.tight_layout()

save_fig("policy_ranking_summary")
plt.show()


## Outlier gate sweep


In [ ]:
gate_values = [0.010, 0.014, 0.018, 0.022, 0.028, 0.035, 0.050]
gate_sweep_rmse = []
gate_sweep_count = []

for gate in gate_values:
    test_omega, _, _, test_gated_omega, _ = robust_kalman_filter_1d(
        delta_Omega_est,
        q=q_omega,
        r=r_omega,
        gate_threshold=gate,
        inflation=inflation,
        p0=1.0,
    )
    test_b, _, _, test_gated_b, _ = robust_kalman_filter_1d(
        delta_B_est,
        q=q_b,
        r=r_b,
        gate_threshold=gate * 0.55,
        inflation=inflation,
        p0=1.0,
    )

    test_eval = evaluate_policy(
        f"gate_{gate}",
        test_omega,
        test_b,
        true_delta_Omega,
        true_delta_B,
    )

    gate_sweep_rmse.append(test_eval["response_rmse_mean"])
    gate_sweep_count.append(int(np.sum(test_gated_omega) + np.sum(test_gated_b)))

gate_sweep_rmse = np.array(gate_sweep_rmse)
best_gate_idx = int(np.argmin(gate_sweep_rmse))
best_gate = float(gate_values[best_gate_idx])

print(f"Current Ω gate = {gate_omega}")
print(f"Best swept Ω gate = {best_gate}")
print(f"Best swept RMSE = {gate_sweep_rmse[best_gate_idx]:.6f}")


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(gate_values, gate_sweep_rmse, marker="o", linewidth=1, label="mean RMSE")
plt.axvline(gate_omega, linestyle="--", linewidth=1, label=f"current gate = {gate_omega}")
plt.axvline(best_gate, linestyle=":", linewidth=1, label=f"best gate = {best_gate}")
plt.xlabel("Ω outlier gate threshold")
plt.ylabel("mean response RMSE")
plt.title("Noise burst robustness: outlier gate sweep")
plt.legend()
plt.tight_layout()

save_fig("outlier_gate_sweep")
plt.show()


## Save robustness summary


In [ ]:
summary = {
    "notebook": "10_noise_burst_robustness",
    "blocks": int(n_blocks),
    "phase_lock_threshold": float(threshold),
    "burst_window": [int(burst_window[0]), int(burst_window[-1])],
    "outlier_blocks": [int(x) for x in outlier_blocks],
    "true_omega_b_correlation": float(np.corrcoef(true_delta_Omega, true_delta_B)[0,1]),
    "measured_omega_b_correlation": float(np.corrcoef(delta_Omega_est, delta_B_est)[0,1]),
    "noise_model": {
        "base_sigma": 0.04,
        "burst_sigma": 0.105,
        "outlier_blocks": [int(x) for x in outlier_blocks],
    },
    "kalman": {
        "q_omega": float(q_omega),
        "q_b": float(q_b),
        "q_cross": float(q_cross),
        "r_omega": float(r_omega),
        "r_b": float(r_b),
    },
    "robust_gate": {
        "gate_omega": float(gate_omega),
        "gate_b": float(gate_b),
        "inflation": float(inflation),
        "omega_gated_blocks": [int(x) for x in np.where(gated_omega)[0]],
        "b_gated_blocks": [int(x) for x in np.where(gated_b)[0]],
        "best_gate_omega": float(best_gate),
        "best_gate_response_rmse": float(gate_sweep_rmse[best_gate_idx]),
    },
    "ranking": policy_table,
}

with open(f"{RESULTS_DIR}/noise_burst_robustness_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/noise_burst_robustness_summary.json")


## Zip outputs for download from Colab


In [ ]:
!zip -r noise_burst_robustness_outputs.zip figures results


## Control-stack status

Notebook 10 introduces robustness testing:

```text
smooth drift → coupled drift → burst/outlier noise
```

## Interpretation

If robust Kalman wins, measurement reliability is the limiting factor.

If ordinary Kalman still wins, noise bursts are not severe enough or gating is too conservative.

## Next directions

- `11_adaptive_covariance_tuning.ipynb`
- `12_failure_boundary_phase_lock.ipynb`
- `noise-mitigation/01_residual_structure_after_bursts.ipynb`

Guiding rule:

> Robust control begins when measurements can be wrong.
